[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aalbata/ai-labs-for-high-school/blob/main/notebooks/sensor_anomaly_detection.ipynb)

# Sensor Data and Anomaly Detection

Hands-On AI for High School, grades 9 to 12

Dr. Ali Al Bataineh, Higher Colleges of Technology, aalbataineh@hct.ac.ae

A factory machine is starting to break. Can you catch the problem in its sensor readings before it fails completely?

You are looking after a spinning machine, something like a large motor or pump. Three sensors are attached to it:

- a vibration sensor, which measures how much the machine shakes
- a temperature sensor, which measures how hot it is getting
- a speed sensor, which measures how fast the shaft turns, in RPM (rotations per minute)

The sensors took a reading every 0.1 seconds for 50 seconds, which gives 500 readings from each sensor. Somewhere in that data, a part called a bearing begins to wear out. Your task is to find where the failure starts without being told where to look.

What you will do:

1. Load the sensor data and look at the numbers.
2. Plot the three signals and make your own guess about when things go wrong.
3. Build a simple algorithm that finds the problem automatically.
4. Compare your guess with the algorithm.
5. Make the algorithm more or less sensitive and see what changes.
6. Combine all three sensors to catch the problem sooner.
7. Compare your version with the systems real factories use.

Every code cell is already written. Click the play button on the left of a cell, wait until it finishes, and then read the output and answer the questions in the text cells. You do not have to write any code, but the code is short and worth reading.

To keep your answers, choose File, then Save a copy in Drive, before you start.

## Step 1: Load the data

The data is stored inside this notebook, so there is nothing to upload. Each reading has four values:

- time, in seconds
- vibration, in g (1 g is the acceleration of gravity, about 9.8 meters per second squared)
- temperature, in degrees Celsius
- RPM, the turning speed of the shaft

Run the cell and read the summary. Look at the smallest, largest and average value for each sensor. Does anything already look unusual?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#fafaf8',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})

# The sensor data is built into this notebook, so there is nothing to upload.
import json
raw = '{"time":[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0,1.1,1.2,1.3,1.4,1.5,1.6,1.7,1.8,1.9,2.0,2.1,2.2,2.3,2.4,2.5,2.6,2.7,2.8,2.9,3.0,3.1,3.2,3.3,3.4,3.5,3.6,3.7,3.8,3.9,4.0,4.1,4.2,4.3,4.4,4.5,4.6,4.7,4.8,4.9,5.0,5.1,5.2,5.3,5.4,5.5,5.6,5.7,5.8,5.9,6.0,6.1,6.2,6.3,6.4,6.5,6.6,6.7,6.8,6.9,7.0,7.1,7.2,7.3,7.4,7.5,7.6,7.7,7.8,7.9,8.0,8.1,8.2,8.3,8.4,8.5,8.6,8.7,8.8,8.9,9.0,9.1,9.2,9.3,9.4,9.5,9.6,9.7,9.8,9.9,10.0,10.1,10.2,10.3,10.4,10.5,10.6,10.7,10.8,10.9,11.0,11.1,11.2,11.3,11.4,11.5,11.6,11.7,11.8,11.9,12.0,12.1,12.2,12.3,12.4,12.5,12.6,12.7,12.8,12.9,13.0,13.1,13.2,13.3,13.4,13.5,13.6,13.7,13.8,13.9,14.0,14.1,14.2,14.3,14.4,14.5,14.6,14.7,14.8,14.9,15.0,15.1,15.2,15.3,15.4,15.5,15.6,15.7,15.8,15.9,16.0,16.1,16.2,16.3,16.4,16.5,16.6,16.7,16.8,16.9,17.0,17.1,17.2,17.3,17.4,17.5,17.6,17.7,17.8,17.9,18.0,18.1,18.2,18.3,18.4,18.5,18.6,18.7,18.8,18.9,19.0,19.1,19.2,19.3,19.4,19.5,19.6,19.7,19.8,19.9,20.0,20.1,20.2,20.3,20.4,20.5,20.6,20.7,20.8,20.9,21.0,21.1,21.2,21.3,21.4,21.5,21.6,21.7,21.8,21.9,22.0,22.1,22.2,22.3,22.4,22.5,22.6,22.7,22.8,22.9,23.0,23.1,23.2,23.3,23.4,23.5,23.6,23.7,23.8,23.9,24.0,24.1,24.2,24.3,24.4,24.5,24.6,24.7,24.8,24.9,25.0,25.1,25.2,25.3,25.4,25.5,25.6,25.7,25.8,25.9,26.0,26.1,26.2,26.3,26.4,26.5,26.6,26.7,26.8,26.9,27.0,27.1,27.2,27.3,27.4,27.5,27.6,27.7,27.8,27.9,28.0,28.1,28.2,28.3,28.4,28.5,28.6,28.7,28.8,28.9,29.0,29.1,29.2,29.3,29.4,29.5,29.6,29.7,29.8,29.9,30.0,30.1,30.2,30.3,30.4,30.5,30.6,30.7,30.8,30.9,31.0,31.1,31.2,31.3,31.4,31.5,31.6,31.7,31.8,31.9,32.0,32.1,32.2,32.3,32.4,32.5,32.6,32.7,32.8,32.9,33.0,33.1,33.2,33.3,33.4,33.5,33.6,33.7,33.8,33.9,34.0,34.1,34.2,34.3,34.4,34.5,34.6,34.7,34.8,34.9,35.0,35.1,35.2,35.3,35.4,35.5,35.6,35.7,35.8,35.9,36.0,36.1,36.2,36.3,36.4,36.5,36.6,36.7,36.8,36.9,37.0,37.1,37.2,37.3,37.4,37.5,37.6,37.7,37.8,37.9,38.0,38.1,38.2,38.3,38.4,38.5,38.6,38.7,38.8,38.9,39.0,39.1,39.2,39.3,39.4,39.5,39.6,39.7,39.8,39.9,40.0,40.1,40.2,40.3,40.4,40.5,40.6,40.7,40.8,40.9,41.0,41.1,41.2,41.3,41.4,41.5,41.6,41.7,41.8,41.9,42.0,42.1,42.2,42.3,42.4,42.5,42.6,42.7,42.8,42.9,43.0,43.1,43.2,43.3,43.4,43.5,43.6,43.7,43.8,43.9,44.0,44.1,44.2,44.3,44.4,44.5,44.6,44.7,44.8,44.9,45.0,45.1,45.2,45.3,45.4,45.5,45.6,45.7,45.8,45.9,46.0,46.1,46.2,46.3,46.4,46.5,46.6,46.7,46.8,46.9,47.0,47.1,47.2,47.3,47.4,47.5,47.6,47.7,47.8,47.9,48.0,48.1,48.2,48.3,48.4,48.5,48.6,48.7,48.8,48.9,49.0,49.1,49.2,49.3,49.4,49.5,49.6,49.7,49.8,49.9],"vibration_g":[0.0051,0.204,0.1837,-0.0765,-0.0447,-0.0791,0.0855,-0.0084,0.112,-0.2771,0.235,-0.0145,0.1021,-0.0205,-0.0569,0.0695,0.1237,-0.0304,-0.0229,0.1029,-0.1306,-0.2272,0.0592,-0.1006,-0.2881,-0.1221,-0.0701,-0.179,-0.2239,0.0055,0.1346,-0.035,-0.1115,0.0577,0.1076,-0.045,0.0817,0.1564,-0.031,-0.122,0.0521,0.0371,0.1648,-0.1927,-0.0992,-0.1257,-0.2601,0.019,0.0792,-0.1108,0.2078,0.1233,0.0941,0.0603,0.1434,-0.1998,0.0921,0.0904,-0.2652,0.0521,-0.0376,0.1172,-0.0659,-0.0027,0.0514,-0.1314,0.0898,-0.0157,0.0739,-0.0783,0.1629,0.0908,-0.0267,0.0948,0.189,0.2687,-0.236,0.1325,0.0698,-0.0141,-0.151,0.1886,-0.1893,0.085,0.1953,-0.24,-0.0454,-0.1964,0.0366,0.2272,0.3035,-0.2667,-0.0862,0.1055,0.2369,0.0632,-0.1119,0.0446,-0.0025,-0.0306,-0.1102,0.0581,0.0462,-0.0139,-0.0333,-0.1927,-0.0729,0.181,-0.0286,-0.216,0.2002,0.0795,0.3162,0.0094,-0.0692,-0.2171,0.1986,0.3854,-0.1231,-0.0971,0.0894,-0.1246,-0.0406,-0.0525,0.0288,0.1642,0.0033,0.1378,-0.063,0.0492,-0.3207,-0.2175,0.1194,-0.0885,0.087,0.0814,0.1983,0.1218,0.1525,-0.0168,-0.1047,-0.1097,-0.0732,-0.1695,-0.0821,-0.0139,0.0377,-0.0508,-0.2886,-0.0108,0.0338,0.1627,0.0867,-0.0965,-0.1086,0.3016,0.1135,0.2747,0.3194,-0.1227,0.0578,0.0687,0.0839,0.0813,0.0303,0.0261,-0.2254,-0.0248,-0.1121,0.0189,-0.0701,0.0928,0.1229,0.0463,0.0474,0.0139,-0.0672,-0.0247,-0.0743,0.0582,0.0021,0.0872,-0.1993,0.1332,-0.1144,-0.1101,-0.0296,-0.0845,0.0437,-0.0861,-0.1604,-0.1269,0.1968,0.0066,-0.1751,0.0013,-0.2334,0.2684,-0.2287,0.0718,0.0815,-0.2175,0.045,0.3199,-0.6506,-0.6712,-0.436,0.0241,0.0505,-0.019,0.0948,-0.1412,0.1188,-0.0758,-0.1636,0.0548,0.2539,0.1443,-0.0773,0.1047,-0.0681,-0.0186,0.0134,-0.3476,0.0287,-0.1544,-0.1046,-0.2212,-0.0649,-0.0054,-0.219,0.5893,0.3572,0.2433,-0.0396,-0.2868,0.0225,0.0668,-0.0643,0.0453,0.0859,-0.1295,-0.2215,-0.0332,-0.0317,-0.0529,0.1481,0.2588,-0.0625,0.1048,0.1411,0.107,0.1571,-0.0584,0.111,0.3037,0.1205,-0.093,0.0912,0.1894,0.0553,-0.0845,0.2315,-0.1875,0.0755,0.5168,-0.3227,0.1264,0.2886,-0.192,0.4872,-0.0647,-0.2854,-0.1033,0.0395,0.0954,0.0248,0.0068,0.0705,-0.0316,0.1765,0.0223,0.0441,0.0811,-0.1586,-0.1064,0.2293,0.0503,-0.0424,0.0512,-0.0722,0.0418,-0.0999,0.0414,-0.2367,0.1996,-0.0769,-0.2372,-0.0336,-0.0556,0.0229,-0.1718,0.0596,0.5435,-0.1918,0.0507,-0.0466,0.0289,-0.2718,-0.1727,0.0689,-0.0058,0.2453,-0.1057,0.0262,0.437,-0.1148,-0.1356,-0.0053,-0.0067,0.2275,0.6138,-0.0459,0.3339,1.0221,0.2093,-0.3168,0.4068,0.6113,0.2405,0.1555,0.338,-0.5355,0.793,-0.1286,-1.1232,-0.047,-1.0319,-0.6633,-0.0364,-0.5423,0.9213,-1.2708,0.1208,0.4342,1.2987,0.4925,0.3961,0.1772,-0.0857,1.4191,-0.6663,0.6708,-0.4713,0.2705,-0.2357,2.9875,-0.5712,0.5674,0.2188,0.5373,0.7459,0.6408,1.6247,-2.2886,0.0667,-2.1553,0.461,-1.9865,-1.5304,-0.0131,2.4197,1.0859,0.5516,2.6913,-1.6494,-3.9906,-0.8726,-0.2525,2.4434,4.4539,-2.6131,5.0467,3.1858,1.8248,0.8594,5.4111,0.0185,-1.1193,0.333,-2.8772,0.7783,-0.1603,0.7426,0.9704,-2.6335,3.3539,-2.277,2.3451,-3.9932,1.0592,-2.2312,0.2337,-1.744,-6.02,-3.0301,0.1987,1.909,-4.1356,-6.1347,-2.2665,-0.4261,-1.6383,-2.2836,2.5989,2.9302,2.2903,-2.727,0.3047,-0.7874,-4.5244,2.6878,0.3618,-1.7253,4.5678,2.6281,2.9335,3.8952,2.6343,-4.0131,-1.53,3.8675,-2.787,0.1121,1.4663,5.0982,5.0279,3.4605,-0.7566,3.4608,1.8324,-0.6189,5.434,1.7192,-2.9767,5.0757,-0.8065,2.6343,-6.1763,-0.1399,-3.7626,-4.0235,-7.9471,9.1893,-0.3334,1.0976,4.7583,-0.3949,1.9894,-5.8984,0.7481,-1.4326,2.3828,0.8757,-6.2508,-8.7621,-4.5009,3.683,1.9029,2.0245,-9.3442,-12.9159,-2.7932,-1.3563,-2.8416,1.7415,-0.6874,-6.0261,-4.3063,0.8943,0.2006,-0.2023,11.4314,-1.6293,0.6179,-5.258,-1.7963,2.5696,3.1417,-0.9295,2.1562,0.2125,-10.1263,-17.4707,12.5588,4.4692,-0.4988,-9.4026,-6.9562,7.5034],"temperature_c":[64.39,64.94,65.2,65.05,65.19,65.0,65.55,64.81,64.74,65.1,65.36,65.04,65.28,64.94,64.8,65.37,65.41,65.41,64.98,64.83,64.83,65.04,65.22,65.21,64.78,65.15,65.11,65.13,65.11,65.08,65.25,65.16,65.18,64.98,65.23,65.12,64.68,65.26,65.1,65.36,65.13,65.15,65.22,65.17,65.08,64.79,65.59,64.84,65.12,65.34,65.29,65.33,65.38,65.49,65.29,65.86,65.01,65.28,65.16,65.12,65.31,65.57,65.1,65.1,65.42,65.38,65.7,65.5,65.33,65.21,65.33,65.25,64.99,65.3,65.26,65.25,65.21,65.68,65.47,65.53,65.19,65.1,65.27,65.39,65.26,65.3,65.56,65.42,65.14,65.27,65.22,65.41,65.12,65.34,65.53,65.39,65.19,65.26,65.42,65.51,65.59,65.38,65.12,65.73,65.49,65.51,65.5,65.2,65.37,65.5,65.67,65.46,64.77,65.4,65.54,65.75,65.62,66.02,65.79,65.78,65.69,65.18,65.68,65.63,65.38,65.63,65.6,65.41,65.49,65.46,65.57,65.43,65.65,65.11,65.48,65.17,65.44,65.67,65.56,65.54,65.58,65.91,65.33,65.77,65.25,65.48,65.62,65.4,65.64,65.76,65.34,65.59,65.53,65.19,65.58,65.45,65.92,65.44,66.01,65.52,66.07,65.77,65.75,65.83,65.86,65.97,66.05,65.6,65.4,65.09,65.6,65.86,65.78,65.72,65.62,65.56,65.65,65.8,65.33,65.88,65.4,65.69,65.59,65.65,65.51,65.49,65.7,65.51,66.3,65.67,65.62,65.65,65.53,65.23,65.72,65.85,65.46,65.79,65.53,65.79,65.88,65.91,65.91,65.66,66.01,65.72,65.9,65.99,65.77,65.56,65.98,65.6,65.65,65.61,66.04,65.83,65.37,66.03,66.33,66.27,65.76,65.9,66.0,66.09,65.91,65.66,66.07,66.05,65.62,65.84,65.98,65.93,66.04,66.11,66.06,65.92,65.65,66.08,66.26,66.39,65.95,65.96,66.29,66.29,66.05,66.14,65.82,66.12,65.81,65.97,65.97,65.9,65.54,65.83,66.09,66.01,65.94,66.12,66.18,66.22,65.86,66.05,65.98,65.86,65.73,66.04,65.87,66.54,66.05,66.03,65.99,66.5,65.88,66.06,65.93,66.07,66.1,66.35,65.8,66.11,66.48,66.01,66.09,65.94,65.56,66.12,66.4,66.05,65.75,66.21,66.18,66.38,66.37,66.11,66.26,65.78,66.43,66.23,66.43,66.43,66.14,66.17,66.02,65.76,66.19,66.25,65.99,66.38,66.25,66.34,66.07,65.86,66.42,66.57,66.59,66.71,66.92,67.14,67.24,67.09,67.89,67.58,68.02,68.2,68.51,68.29,69.09,69.15,69.36,69.66,69.39,69.06,70.0,70.4,70.55,70.36,70.65,70.91,70.64,71.07,71.26,71.93,71.63,72.3,72.14,71.95,73.0,72.48,73.01,73.44,73.13,73.16,73.84,73.81,73.71,73.93,74.26,74.74,74.65,75.04,75.45,75.55,75.77,75.91,75.47,76.27,76.26,76.34,76.32,76.39,76.68,77.1,77.03,77.3,77.52,77.99,77.96,78.18,78.69,78.51,78.9,78.72,79.2,79.34,79.54,79.58,79.64,79.99,80.13,80.65,80.8,80.52,81.11,81.05,81.44,81.42,81.38,82.12,82.18,81.99,82.03,82.51,82.55,82.92,83.32,83.69,83.47,84.04,84.17,84.0,83.92,84.66,84.79,85.35,85.33,85.38,85.76,85.34,85.8,85.88,86.51,86.43,85.82,86.98,86.66,87.22,87.12,87.47,87.74,87.83,87.76,88.46,88.42,88.52,88.86,88.77,89.49,89.31,89.52,89.7,89.61,89.73,90.22,90.44,90.84,90.83,90.67,91.21,91.46,91.68,91.72,91.56,92.27,92.04,92.51,92.22,92.92,92.76,93.36,93.69,93.75,93.42,93.65,93.85,94.37,94.48,94.63,94.84,95.09,95.29,95.7,95.5,95.74,95.47,95.77,96.09,96.17,96.66,97.03,96.89,97.06,97.24,97.56,97.6,98.0,98.15,98.35,98.62,98.64,98.85,99.39,99.18,99.55,99.57,99.64,99.82,99.93,100.29,100.34,100.57],"rpm":[1804.9,1799.5,1804.6,1801.1,1803.4,1797.5,1800.6,1810.5,1803.3,1804.0,1800.9,1803.4,1795.3,1796.6,1801.7,1803.3,1798.3,1799.2,1798.5,1799.2,1795.2,1794.2,1799.6,1793.0,1796.0,1797.3,1795.7,1800.9,1798.2,1799.6,1795.9,1797.0,1802.0,1801.4,1798.1,1801.4,1800.7,1800.5,1806.3,1798.3,1801.6,1801.6,1798.9,1795.2,1802.6,1800.9,1800.1,1800.0,1799.9,1796.3,1801.0,1792.6,1799.7,1793.9,1798.4,1802.7,1798.8,1797.9,1801.9,1798.7,1798.2,1798.2,1805.8,1802.5,1801.7,1795.0,1802.3,1794.6,1801.7,1795.3,1807.6,1793.0,1804.7,1803.0,1792.7,1798.8,1805.5,1804.3,1793.1,1797.7,1795.1,1795.7,1797.2,1807.1,1800.3,1798.8,1798.4,1802.4,1802.5,1796.1,1805.6,1795.4,1798.6,1801.5,1806.6,1798.4,1800.9,1805.0,1802.2,1804.2,1803.9,1800.8,1802.3,1803.4,1799.6,1806.1,1791.8,1796.1,1805.5,1801.4,1797.2,1801.3,1804.1,1797.5,1802.8,1799.3,1799.3,1799.5,1804.2,1795.3,1798.3,1802.4,1794.9,1793.4,1802.0,1799.8,1799.2,1800.3,1795.9,1811.4,1795.8,1798.0,1794.1,1798.1,1805.9,1799.8,1802.1,1797.8,1807.0,1800.4,1794.8,1802.8,1798.3,1802.2,1800.3,1800.8,1800.3,1796.1,1793.7,1803.6,1799.2,1800.9,1797.9,1791.7,1794.6,1805.3,1796.6,1798.2,1793.9,1797.6,1798.3,1798.9,1799.4,1797.2,1797.4,1799.7,1801.0,1799.2,1806.7,1800.6,1800.4,1797.1,1803.8,1795.6,1793.3,1795.8,1802.2,1795.1,1804.9,1799.6,1793.7,1798.8,1806.9,1796.3,1794.5,1801.8,1804.9,1803.8,1798.4,1800.5,1801.0,1804.4,1808.3,1799.3,1804.9,1798.3,1802.1,1796.0,1796.7,1801.4,1796.1,1803.3,1797.5,1800.0,1799.4,1798.4,1800.2,1804.0,1812.1,1795.0,1800.9,1799.0,1796.1,1800.9,1796.2,1799.4,1804.7,1791.6,1800.7,1801.5,1795.9,1801.8,1801.1,1798.3,1804.6,1794.5,1802.3,1798.8,1795.5,1799.0,1798.3,1803.0,1793.7,1798.9,1797.9,1806.0,1795.2,1803.9,1792.7,1801.1,1799.2,1801.2,1796.9,1791.9,1793.5,1804.9,1800.5,1796.2,1803.3,1796.7,1796.7,1794.8,1804.4,1807.5,1795.8,1800.8,1803.6,1808.1,1790.1,1800.2,1802.2,1793.7,1801.3,1800.4,1802.5,1797.3,1801.3,1797.2,1802.0,1800.3,1798.1,1794.0,1806.2,1804.1,1803.1,1797.0,1798.2,1806.3,1802.6,1796.2,1799.8,1800.2,1798.8,1795.9,1804.6,1802.0,1798.8,1793.3,1797.3,1799.6,1803.1,1798.2,1804.1,1800.2,1796.7,1788.5,1795.0,1793.7,1799.1,1801.0,1805.2,1799.6,1795.1,1809.1,1800.9,1798.6,1798.6,1794.3,1802.2,1801.2,1806.1,1804.1,1803.2,1804.8,1798.6,1798.0,1799.5,1794.7,1799.2,1798.4,1802.4,1799.7,1792.1,1799.9,1805.4,1792.4,1796.7,1799.8,1794.5,1797.6,1800.0,1794.5,1801.0,1800.4,1792.7,1796.9,1800.1,1794.1,1795.6,1795.4,1790.9,1794.8,1791.4,1784.9,1789.9,1790.3,1789.1,1782.3,1785.7,1788.3,1782.8,1785.6,1785.1,1778.7,1780.5,1775.7,1780.2,1774.5,1771.7,1772.8,1771.6,1772.4,1771.8,1764.2,1765.3,1768.3,1767.8,1765.3,1774.5,1767.4,1761.6,1763.8,1765.3,1760.4,1751.0,1757.3,1756.2,1763.5,1746.6,1759.2,1751.4,1754.7,1748.8,1749.7,1752.5,1747.3,1750.2,1744.7,1742.4,1742.2,1740.0,1740.6,1745.5,1734.4,1731.4,1737.0,1742.8,1739.2,1739.3,1738.1,1729.1,1731.5,1719.4,1726.6,1730.5,1720.8,1717.8,1719.9,1725.0,1727.8,1732.4,1726.8,1714.1,1709.8,1718.7,1721.4,1709.7,1713.1,1712.4,1713.3,1711.5,1713.3,1714.4,1709.5,1711.2,1715.2,1702.1,1701.0,1706.3,1710.7,1702.9,1705.6,1699.0,1702.9,1695.8,1700.0,1697.9,1696.9,1699.8,1697.6,1689.9,1694.9,1683.6,1693.3,1699.4,1689.5,1688.4,1685.7,1687.7,1681.6,1685.1,1680.4,1678.8,1680.0,1678.2,1670.5,1684.8,1676.0,1667.7,1677.8,1672.4,1672.3,1673.3,1665.9,1656.7,1667.1,1667.5,1669.5,1661.5,1662.4,1664.1,1660.2,1659.5,1660.5,1654.5,1663.8,1652.8,1655.6,1653.2,1650.7,1654.6,1655.5,1656.3,1651.8,1646.7,1650.9,1640.7,1645.7,1646.6,1645.7,1637.9,1635.5,1647.1,1637.4,1631.9,1632.5,1636.5,1640.1,1634.4,1638.2]}'
data = json.loads(raw)

time = np.array(data['time'])
vib  = np.array(data['vibration_g'])
temp = np.array(data['temperature_c'])
rpm  = np.array(data['rpm'])

print('Data loaded successfully')
print('=' * 50)
print(f'Total readings:    {len(time)}')
print(f'One reading every: 0.1 seconds')
print(f'Time covered:      {len(time) * 0.1:.0f} seconds')
print()
print('Signal summary:')
print(f'  Vibration:   min={vib.min():.2f}g   max={vib.max():.2f}g   mean={vib.mean():.3f}g')
print(f'  Temperature: min={temp.min():.1f}C  max={temp.max():.1f}C  mean={temp.mean():.1f}C')
print(f'  RPM:         min={rpm.min():.0f}    max={rpm.max():.0f}    mean={rpm.mean():.0f}')

### Checkpoint 1

Double-click this cell to edit it, then write your prediction.

**Looking only at the summary numbers above, does anything look suspicious to you? What would you watch for?**

*Write your answer here.*

## Step 2: Look at the three signals

Before using any algorithm, look at the data. Your eyes are good at spotting the moment something changes.

When the machine is healthy, you would expect:

- vibration: a steady amount of shaking, evenly above and below zero
- temperature: a slow, gentle rise as the machine warms up
- RPM: a steady speed close to its target, with only small wobbles

Run the cell and study the three plots for a minute before you answer.

In [ ]:
fig = plt.figure(figsize=(14, 9))
gs = GridSpec(3, 1, figure=fig, hspace=0.45)

# Vibration
ax1 = fig.add_subplot(gs[0])
ax1.plot(time, vib, color='#6B1E2B', linewidth=0.7, alpha=0.85, label='Vibration (g)')
ax1.axhline(y=0, color='#999', linewidth=0.5, linestyle='--')
ax1.set_ylabel('Vibration (g)', fontsize=11)
ax1.set_title('Machine Sensor Data: 500 Readings over 50 Seconds', fontsize=13, fontweight='bold', pad=10)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_xlim(0, 50)

# Temperature
ax2 = fig.add_subplot(gs[1])
ax2.plot(time, temp, color='#B8962E', linewidth=1.2, label='Temperature (C)')
ax2.set_ylabel('Temperature (C)', fontsize=11)
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xlim(0, 50)

# RPM
ax3 = fig.add_subplot(gs[2])
ax3.plot(time, rpm, color='#1a2744', linewidth=1.0, label='RPM')
ax3.axhline(y=1800, color='#999', linewidth=0.8, linestyle='--', alpha=0.6, label='Target RPM (1800)')
ax3.set_ylabel('RPM', fontsize=11)
ax3.set_xlabel('Time (seconds)', fontsize=11)
ax3.legend(loc='upper right', fontsize=9)
ax3.set_xlim(0, 50)

plt.show()
print('Study all three signals carefully before you answer Checkpoint 2.')

### Checkpoint 2: Reading the signals

Double-click to edit and write your answers.

**1. At about what time (in seconds) do you think the trouble begins? Write your best guess.**

*My guess: t = _____ seconds*

**2. Which signal gave you the clearest warning: vibration, temperature, or RPM?**

*Write here.*

**3. In your own words, what happens to each signal over the 50 seconds?**
- Vibration: *Write here.*
- Temperature: *Write here.*
- RPM: *Write here.*

**4. If you had to decide right now, based only on this plot, would you stop the machine or keep it running? Why?**

*Write here.*

## Step 3: Zoom in

A plot of all 50 seconds can hide detail. This cell zooms in on the vibration signal in two short windows, one where the machine looks healthy and one where something looks wrong.

To compare them with a number, it uses the standard deviation, a measure of how spread out, or jumpy, a signal is. A calm signal has a low standard deviation and a jumpy one has a high standard deviation.

You can change the window start times and run the cell again to look at other parts of the signal.

In [ ]:
# Change these to zoom into different parts of the signal
WINDOW_A_START = 5.0   # seconds, a normal window
WINDOW_A_END   = 10.0
WINDOW_B_START = 38.0  # seconds, a suspected problem window
WINDOW_B_END   = 43.0

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, t_start, t_end, label, color in [
    (axes[0], WINDOW_A_START, WINDOW_A_END, 'Normal Window', '#6B1E2B'),
    (axes[1], WINDOW_B_START, WINDOW_B_END, 'Suspected Problem Window', '#c0392b'),
]:
    mask = (time >= t_start) & (time <= t_end)
    t_win = time[mask]
    v_win = vib[mask]
    ax.plot(t_win, v_win, color=color, linewidth=1.2)
    ax.fill_between(t_win, v_win, alpha=0.12, color=color)
    ax.axhline(0, color='#999', linewidth=0.5, linestyle='--')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Vibration (g)')
    ax.text(0.02, 0.95, f'Std dev: {v_win.std():.3f} g', transform=ax.transAxes,
            fontsize=10, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.suptitle('Vibration Signal: Zoomed Windows', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

mask_a = (time >= WINDOW_A_START) & (time <= WINDOW_A_END)
mask_b = (time >= WINDOW_B_START) & (time <= WINDOW_B_END)
std_a = vib[mask_a].std()
std_b = vib[mask_b].std()
print(f'Window A ({WINDOW_A_START}-{WINDOW_A_END}s) vibration std dev: {std_a:.4f} g')
print(f'Window B ({WINDOW_B_START}-{WINDOW_B_END}s) vibration std dev: {std_b:.4f} g')
print(f'The problem window is {std_b/std_a:.1f} times jumpier than the normal window.')

## Step 4: Detect the problem automatically

Watching a screen works, but it is slow, and someone has to be watching. A real monitoring system runs day and night, so it needs an algorithm that raises the alarm by itself.

Here is a simple idea that works, called a rolling standard deviation. Slide a short window along the signal (here, the last 30 readings, about 3 seconds). At each point, measure how jumpy the signal is inside the window. When the jumpiness rises above a level you choose, called the threshold, flag a problem.

Seismologists use a similar idea to detect earthquakes. A station compares its average signal over the last fraction of a second with its average over a longer period, and a sudden jump in that ratio marks the arrival of an earthquake wave.

The one choice you make here is the threshold: how far above normal counts as a real problem.

In [ ]:
# PARAMETERS: change these and run again to see the effect
WINDOW_SIZE      = 30    # how many readings in the rolling window (30 = about 3 seconds)
THRESHOLD_FACTOR = 2.5   # flag if jumpiness goes above this many times the normal level

def rolling_std(signal, window):
    """Measure how jumpy the signal is over a sliding window of recent readings."""
    result = np.full(len(signal), np.nan)
    for i in range(window, len(signal)):
        result[i] = signal[i-window:i].std()
    return result

# Jumpiness of the vibration signal over time
roll_std = rolling_std(vib, WINDOW_SIZE)

# Normal level: the average jumpiness during the first 15 seconds (150 readings)
baseline_std = np.nanmean(roll_std[:150])
threshold = baseline_std * THRESHOLD_FACTOR

# Flag anything above the threshold
anomaly_flags = roll_std > threshold
anomaly_times = time[anomaly_flags]
first_detection = anomaly_times[0] if len(anomaly_times) > 0 else None

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Top: the vibration signal, with flagged regions shaded
ax1.plot(time, vib, color='#6B1E2B', linewidth=0.6, alpha=0.8, label='Vibration')
ax1.fill_between(time, vib, where=anomaly_flags, color='#c0392b', alpha=0.25, label='Flagged as a problem')
if first_detection:
    ax1.axvline(first_detection, color='#c0392b', linewidth=2, linestyle='--',
                label=f'First alarm: t={first_detection:.1f}s')
ax1.set_ylabel('Vibration (g)', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_title(f'Automatic Detection  |  Window: {WINDOW_SIZE} readings  |  Threshold: {THRESHOLD_FACTOR}x normal',
               fontsize=11, fontweight='bold')

# Bottom: the jumpiness over time, with the threshold line
ax2.plot(time, roll_std, color='#1a2744', linewidth=1.2, label=f'Jumpiness ({WINDOW_SIZE}-reading window)')
ax2.axhline(threshold, color='#c0392b', linewidth=1.8, linestyle='--',
             label=f'Threshold ({THRESHOLD_FACTOR}x normal = {threshold:.3f}g)')
ax2.axhline(baseline_std, color='#B8962E', linewidth=1, linestyle=':', alpha=0.8,
             label=f'Normal level ({baseline_std:.3f}g)')
ax2.fill_between(time, roll_std, threshold, where=(roll_std > threshold), color='#c0392b', alpha=0.2)
ax2.set_ylabel('Jumpiness (g)', fontsize=11)
ax2.set_xlabel('Time (seconds)', fontsize=11)
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xlim(0, 50)

plt.tight_layout()
plt.show()

true_start = 32.0  # the real answer: the bearing fault begins here
print('Detection results')
print('=' * 50)
print(f'Normal level of jumpiness: {baseline_std:.4f} g')
print(f'Alarm threshold:           {threshold:.4f} g  ({THRESHOLD_FACTOR}x normal)')
print(f'Algorithm first alarm:     t = {first_detection:.1f}s' if first_detection else 'No problem detected')
print(f'True start of the fault:   t = {true_start:.1f}s')
if first_detection:
    print(f'The alarm came {first_detection - true_start:.1f}s after the fault actually began.')

### Checkpoint 3: Detection results

**1. How close was your guess from Checkpoint 2 to the algorithm's first alarm?**

*Write here.*

**2. How close was the algorithm's first alarm to the true start of the fault (t = 32.0s)?**

*Write here.*

**3. The alarm came a little while after the fault really began. In a real machine, what could happen during that delay?**

*Write here.*

## Step 5: The sensitivity trade-off

The threshold is the most important setting, so this step changes it.

- Set it too high and the alarm stays quiet too long, so you learn about the failure late.
- Set it too low and ordinary bumps set off the alarm, which gives false alarms.

This is the same trade-off as in the object-recognition lab. Catching every real failure and never raising a false alarm pull in opposite directions, and you cannot fully have both. False alarms cost money, because a machine is shut down for nothing. Missed failures can be dangerous, because the machine breaks without warning.

Picture a pump that supplies drinking water to a town of 50,000 people. Which mistake would you rather make?

In [ ]:
# Compare several threshold settings side by side
thresholds_to_test = [1.5, 2.0, 2.5, 3.5, 5.0]
true_start = 32.0

print('Threshold sensitivity')
print('=' * 70)
print(f'{"Threshold":>12}  {"First alarm":>16}  {"Delay":>8}  {"False alarms":>14}  {"Verdict":>14}')
print('-' * 70)

for tf in thresholds_to_test:
    t_val = baseline_std * tf
    flags = roll_std > t_val

    # False alarms: anything flagged before the fault truly began
    false_alarms = int(flags[time < true_start].sum())

    # First alarm at or after the true start
    post_start = time[(flags) & (time >= true_start)]
    first_det = post_start[0] if len(post_start) > 0 else None
    delay = (first_det - true_start) if first_det is not None else None

    delay_str = f'{delay:.1f}s' if delay is not None else 'MISSED'
    det_str   = f't={first_det:.1f}s' if first_det is not None else 'Not detected'

    if false_alarms > 5:
        verdict = 'Too sensitive'
    elif first_det is None:
        verdict = 'Too cautious'
    elif delay < 3:
        verdict = 'Good'
    elif delay < 6:
        verdict = 'Acceptable'
    else:
        verdict = 'Too slow'

    print(f'{tf:>11.1f}x  {det_str:>16}  {delay_str:>8}  {false_alarms:>14}  {verdict:>14}')

print()
print(f'Normal level of jumpiness: {baseline_std:.4f} g')
print(f'True start of the fault:   t = {true_start}s')
print()
print('Which threshold would you choose for each of these machines?')
print('  A) A coolant pump in a nuclear power plant')
print('  B) A conveyor belt in a packaging warehouse')
print('  C) A fan motor in an office air-conditioning system')

### Checkpoint 4: Sensitivity

**1. Which threshold gave the best balance for this machine? Why?**

*Write here.*

**2. Choose a threshold for each machine below and explain your reasoning:**
- Nuclear plant coolant pump: threshold = _____ because *write here.*
- Packaging-warehouse conveyor belt: threshold = _____ because *write here.*
- Office air-conditioning fan motor: threshold = _____ because *write here.*

**3. The best threshold for this machine might be wrong for a different machine. What would you want to know about a machine before setting its threshold?**

*Write here.*

## Step 6: Use all three sensors together

So far only vibration has been used. Real monitoring systems usually combine several sensors, because one sensor on its own can mislead you. A single vibration spike might come from a heavy load. If vibration, temperature and speed all change at the same time, that is much stronger evidence of a real problem.

In this lab, temperature counts double in the combined score, because friction in a failing bearing makes it heat up.

In [ ]:
# Put each signal on the same 0 to 1 scale so they can be compared fairly
def normalize(signal):
    return (signal - signal.min()) / (signal.max() - signal.min())

vib_norm  = normalize(np.abs(vib))         # we care about size of shaking, not direction
temp_norm = normalize(temp)
rpm_dev   = normalize(np.abs(rpm - 1800))  # how far speed is from its target

# Jumpiness of each signal
vib_roll  = np.nan_to_num(rolling_std(vib_norm,  WINDOW_SIZE))
temp_roll = np.nan_to_num(rolling_std(temp_norm, WINDOW_SIZE))
rpm_roll  = np.nan_to_num(rolling_std(rpm_dev,   WINDOW_SIZE))

# Combined score: temperature counts double because bearing failures heat up
combined_score = (vib_roll + temp_roll * 2 + rpm_roll * 1.5) / 4.5

baseline_combined = np.mean(combined_score[:150])
combined_threshold = baseline_combined * 3.0
combined_flags = combined_score > combined_threshold
combined_detect_times = time[combined_flags]
combined_first = combined_detect_times[0] if len(combined_detect_times) > 0 else None

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(time, vib_norm,  color='#6B1E2B', linewidth=0.8, alpha=0.8, label='Vibration')
ax1.plot(time, temp_norm, color='#B8962E', linewidth=1.0, alpha=0.8, label='Temperature')
ax1.plot(time, rpm_dev,   color='#1a2744', linewidth=0.8, alpha=0.7, label='Speed away from target')
ax1.set_ylabel('Scaled value (0-1)', fontsize=11)
ax1.legend(loc='upper left', fontsize=9)
ax1.set_title('All Three Signals on the Same Scale', fontsize=11, fontweight='bold')

ax2.plot(time, combined_score, color='#1a2744', linewidth=1.3, label='Combined score')
ax2.axhline(combined_threshold, color='#c0392b', linewidth=1.8, linestyle='--',
             label=f'Threshold ({combined_threshold:.4f})')
ax2.fill_between(time, combined_score, combined_threshold,
                  where=(combined_score > combined_threshold), color='#c0392b', alpha=0.25,
                  label='Combined alarm')
if combined_first:
    ax2.axvline(combined_first, color='#c0392b', linewidth=2, linestyle=':',
                label=f'First combined alarm: t={combined_first:.1f}s')
ax2.set_ylabel('Combined score', fontsize=11)
ax2.set_xlabel('Time (seconds)', fontsize=11)
ax2.legend(loc='upper left', fontsize=9)
ax2.set_xlim(0, 50)

plt.tight_layout()
plt.show()

print('Multi-signal results')
print('=' * 50)
print(f'Vibration only (2.5x):  t={first_detection:.1f}s' if first_detection else 'Not detected')
print(f'All three signals:      t={combined_first:.1f}s' if combined_first else 'Not detected')
print(f'True start of the fault: t=32.0s')
if combined_first and first_detection:
    gap = first_detection - combined_first
    word = 'earlier' if gap > 0 else 'later'
    print(f'Using all three signals caught the problem {abs(gap):.1f}s {word} than vibration alone.')

### Checkpoint 5: Multi-signal

**1. Did combining all three signals catch the problem sooner than vibration alone? By how much?**

*Write here.*

**2. Temperature was counted double in the combined score. Do you agree with that choice? What physical reason supports it?**

*Write here.*

**3. Name two other sensors you would add to this machine if you were designing a real monitoring system, and say what each one would tell you.**

*Write here.*

## Step 7: What real factory systems do

The algorithm you built uses the core idea behind predictive maintenance, which means predicting a failure early enough to fix the machine before it breaks. Companies such as Siemens, GE Vernova and ABB sell predictive-maintenance software. Their systems use many more sensors and more advanced methods, sometimes neural networks trained on large amounts of machine data.

Run the cell to see a side-by-side comparison.

In [ ]:
rows_you = [
    ('Number of sensors',  '3 (vibration, temperature, speed)', 'Many more, of more kinds'),
    ('How it decides',     'Jumpiness above a threshold',        'Often a trained machine-learning model'),
    ('Who sets it up',     'You picked 2.5x by hand',            'Learned from long periods of normal use'),
    ('Faults it knows',    '1 (a worn bearing)',                 'Many kinds of failure'),
    ('False-alarm rate',   'Depends on your threshold',          'Tuned to how critical the machine is'),
    ('Improves over time', 'No',                                 'Can be retrained as the machine ages'),
]
rows_same = [
    ('Main idea',          'Notice when things drift from normal', 'Notice when things drift from normal'),
    ('Key human input',    'Knowing what normal looks like',       'Knowing what normal looks like'),
    ('What people still do','Set the threshold, read the results', 'Check the alerts, decide what to do'),
]

print(f'{"":<22} {"What you built":<38} {"A real factory system":<38}')
print('=' * 98)
for label, yours, real in rows_you:
    print(f'{label:<22} {yours:<38} {real:<38}')
print()
print('  WHAT STAYS THE SAME')
for label, yours, real in rows_same:
    print(f'{label:<22} {yours:<38} {real:<38}')
print()
print('The main point:')
print('Even the most advanced factory AI runs on the same idea you used today,')
print('spotting when a machine drifts away from normal. Deciding what "normal"')
print('means, and which changes actually matter, is still a human job.')

## Final reflection

Double-click this cell and write your answers.

1. Explain in your own words how the rolling-standard-deviation detector works, as if you were telling a factory manager who has never taken a statistics class.

   *Write here.*

2. You used a threshold of 2.5x in Step 4, and the real fault started at t = 32.0 s. How much sooner could a lower threshold have caught it, and what would that cost?

   *Write here.*

3. A worker who walks past this machine every day might notice something wrong before any sensor does. What might they notice, and how is that different from what the algorithm sees?

   *Write here.*

4. This machine had a worn bearing. Name two other ways a machine can break, and describe what you would expect to see in the sensor data for each.

   *Write here.*

5. A factory wants to put this system on 500 machines and text the maintenance team every time an alarm goes off. What could go wrong, and what safety checks would you add?

   *Write here.*

---

Hands-On AI for High School · Dr. Ali Al Bataineh, Higher Colleges of Technology · aalbataineh@hct.ac.ae